# Data Ingestion
---
This notebook retrieves optical, C-Band SAR, and L-Band SAR data for a user specified region of interest.

## Packages and API Authentication
---

In [1]:
import ee
import geemap
import rioxarray as rxr
import pyproj

In [2]:
ee.Authenticate()
ee.Initialize(
    project="multimodal-regression"
)

## Define ROI
---

In [109]:
"""Manually select ROI."""

m = geemap.Map(basemap="SATELLITE")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [110]:
"""Extract bounding box."""
# ee_roi = m.user_roi.bounds()
# ee_roi.coordinates().getInfo()

'Extract bounding box.'

In [3]:
"""Get bounding box from raster."""

# Read in raster from file
src_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/lidar_bc_dems/chm/bc_092p038_3_1_3_CHM_10m.tif"
img = rxr.open_rasterio(src_fp)

# Get 2D bounds
bounds = img.rio.bounds()

xmin, ymin, xmax, ymax = bounds
xmin -= 100
ymin -= 100
xmax += 100
ymax += 100
bounds = (xmin, ymin, xmax, ymax)

cm_crs_str = f"EPSG:{pyproj.CRS(img.rio.crs).to_2d().to_epsg()}"

# Project bounds to WGS84
transformer = pyproj.Transformer.from_crs(cm_crs_str, "EPSG:4326", always_xy=True)
xmin_wgs, ymin_wgs = transformer.transform(bounds[0], bounds[1])
xmax_wgs, ymax_wgs = transformer.transform(bounds[2], bounds[3])

# Create ee geom obj
ee_roi = ee.Geometry.Rectangle([xmin_wgs, ymin_wgs, xmax_wgs, ymax_wgs])

## Optical: Sentinel-2
---

In [9]:
"""Fetch optical data and convert to a local Xarray Dataset."""

# Define processing helper
def process_s2(image):
    """
    Masks clouds and snow  from Sentinel-2 imagery using the
    provided scene classification layer and a probability
    threshold, then returns spectral bands scaled to percent reflectance.
    """
    mask_cld = image.select("MSK_CLDPRB").lt(2)    # pr_cld
    mask_snw = image.select("MSK_SNWPRB").lt(2)    # pr_snw

    spectral_bands = image.select(["B2", "B3", "B4", "B8"])

    return spectral_bands.updateMask(mask_cld).updateMask(mask_snw).divide(10000)


# Filter collection
date_start = "2024-07-20"
date_end = "2024-09-14"
cld_percentage = 5

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_roi)
    .filterDate(date_start, date_end)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cld_percentage))
).sort("CLOUDY_PIXEL_PERCENTAGE")

print(f"Images found: {s2_col.size().getInfo()}")

# Process collection
s2_col_processed = s2_col.map(process_s2)             # pixelwise mask and scale
s2_single = s2_col_processed.first().clip(ee_roi)
s2_img_med = s2_col_processed.median().clip(ee_roi)   # median composite (less noisy for structure analysis)

# Covert to Xarray
native_projection = s2_col.first().select("B2").projection()
s2_img_med_projected = s2_img_med.setDefaultProjection(native_projection)   # force composite projection to native projection

s2_ds = geemap.ee_to_xarray(
    dataset=s2_img_med_projected,
    geometry=ee_roi,
    scale=10,       # 10m native Sentinel-2 resolution
    crs=cm_crs_str,
)

print(s2_ds.info())

Images found: 7
xarray.Dataset {
dimensions:
	time = 1 ;
	y = 168 ;
	x = 213 ;

variables:
	float32 B2(time, y, x) ;
		B2:id = B2 ;
		B2:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B2:dimensions = [14635, 22620] ;
		B2:origin = [-3136, 3472] ;
		B2:crs = EPSG:3157 ;
		B2:crs_transform = [10, 0, 600000, 0, -10, 5800020] ;
	float32 B3(time, y, x) ;
		B3:id = B3 ;
		B3:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B3:dimensions = [14635, 22620] ;
		B3:origin = [-3136, 3472] ;
		B3:crs = EPSG:3157 ;
		B3:crs_transform = [10, 0, 600000, 0, -10, 5800020] ;
	float32 B4(time, y, x) ;
		B4:id = B4 ;
		B4:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B4:dimensions = [14635, 22620] ;
		B4:origin = [-3136, 3472] ;
		B4:crs = EPSG:3157 ;
		B4:crs_transform = [10, 0, 600000, 0, -10, 5800020] ;
	float32 B8(time, y, x) ;
		B8:id = B8 ;
		B8:data_type =

In [10]:
"""Visualize data."""

m = geemap.Map()
true_color_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3
}

false_color_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0,
    "max": 0.3
}

qa_vis = {
    "bands": ["MSK_CLDPRB"],
}

m.centerObject(ee_roi, zoom=12)
m.addLayer(s2_img_med, true_color_vis, "Median True Color (RGB)")
m.addLayer(s2_img_med, false_color_vis, "Median False Color (NIR,R,G)")

m.addLayer(s2_single, true_color_vis, "Single True Color (RGB)")
m.addLayer(s2_single, false_color_vis, "Single False Color (NIR,R,G)")
m

Map(center=[51.3688006359623, -120.58755236306146], controls=(WidgetControl(options=['position', 'transparent_…

In [6]:
"""Write to GeoTiff."""

# Shape dataset dimensions
s2_ds_2d = s2_ds.squeeze("time", drop=True)

# Write
dst_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/s2.tif"
s2_ds_2d.rio.to_raster(
    dst_fp,
    driver="GTiff",
    compress="deflate",   # lossless compression
    tiled=True,           # internal tiles for faster loading in QGIS
)

## C-Band SAR: Sentinel-1
---

In [32]:
"""Fetch C-Band SAR data and convert to a local Xarray Dataset.""" 

# Load Sentinel-1 GRD collection
date_start = "2024-06-26"
date_end = "2024-08-21"

s1_asc_col = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(ee_roi)
    .filterDate(date_start, date_end)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    # .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
).sort("system:time_start")

print(f"Images found: {s1_asc_col.size().getInfo()}")

# Compute median composite and clip
s1_med = s1_asc_col.select(["VV", "VH"]).median().clip(ee_roi)

# Convert GEE image to local Xarray dataset
s1_ds = geemap.ee_to_xarray(
    dataset=s1_med,
    geometry=ee_roi,
    scale=10,
    crs=cm_crs_str,
)

# Inspect the resulting lazy-loaded datacube
print(s1_ds)

Images found: 5
<xarray.Dataset> Size: 289kB
Dimensions:  (time: 1, y: 168, x: 213)
Coordinates:
  * time     (time) int64 8B 0
  * y        (y) float64 1kB 5.694e+06 5.694e+06 ... 5.693e+06 5.693e+06
  * x        (x) float64 2kB 6.669e+05 6.669e+05 6.669e+05 ... 6.69e+05 6.69e+05
Data variables:
    VV       (time, y, x) float32 143kB ...
    VH       (time, y, x) float32 143kB ...


In [25]:
s1_asc_col

In [116]:
"""Inspect data on map."""

m = geemap.Map()
m.add_basemap("SATELLITE")
m.centerObject(ee_roi, zoom=12)
m.add_layer(s1_med.select("VV"), {'min': -25, 'max': 5}, "VV")
m.add_layer(s1_med.select("VH"), {'min': -25, 'max': 5}, "VH")
m

Map(center=[54.16876275368457, -123.68751356198669], controls=(WidgetControl(options=['position', 'transparent…

In [117]:
"""Write to GeoTiff."""

# Shape dataset dimensions
s1_ds_2d = s1_ds.squeeze("time", drop=True)

# Write
dst_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/sar/s1.tif"
s1_ds_2d.rio.to_raster(
    dst_fp,
    driver="GTiff",
    compress="deflate",   # lossless compression
    tiled=True,           # internal tiles for faster loading in QGIS
)

## L-Band SAR: ALOS PALSAR
---

In [118]:
"""Fetch L-Band data, convert to dB, then convert to a local Xarray Dataset."""

year = 2024

palsar_img = ee.Image(f"JAXA/ALOS/PALSAR/YEARLY/SAR_EPOCH/{year}")

# Select bands and clip
palsar_pol = palsar_img.select(["HH", "HV"]).clip(ee_roi)

gamma_naught = palsar_pol.log10().multiply(20).subtract(83.0)

# Convert GEE image to local Xarray dataset
palsar_ds = geemap.ee_to_xarray(
    dataset=gamma_naught,
    geometry=ee_roi,
    scale=10,               # Upsample to match Sentinel-2
    crs=cm_crs_str,
)

print(palsar_ds)

<xarray.Dataset> Size: 232kB
Dimensions:  (time: 1, y: 159, x: 180)
Coordinates:
  * time     (time) int64 8B 0
  * y        (y) float64 1kB 6.003e+06 6.003e+06 ... 6.002e+06 6.002e+06
  * x        (x) float64 1kB 4.542e+05 4.542e+05 4.542e+05 ... 4.56e+05 4.56e+05
Data variables:
    HH       (time, y, x) float32 114kB ...
    HV       (time, y, x) float32 114kB ...


In [119]:
"""Inspect data on map."""

m = geemap.Map()
m.add_basemap("SATELLITE")

m.add_layer(gamma_naught.select(['HH']), {'min': -25, 'max': 5}, 'HH polarization')
m.add_layer(gamma_naught.select(['HV']), {'min': -25, 'max': 5}, 'HV polarization')
m.center_object(gamma_naught)
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [120]:
"""Write to GeoTiff."""

# Shape dataset dimensions
palsar_ds_2d = palsar_ds.squeeze("time", drop=True)

# Write
dst_fp = "/mnt/c/Users/sebas/Documents/projects/multimodal-canopy/data/sar/alos.tif"
palsar_ds_2d.rio.to_raster(
    dst_fp,
    driver="GTiff",
    compress="deflate",   # lossless compression
    tiled=True,           # internal tiles for faster loading in QGIS
)